# Phase 1 · Taxonomy and host identities

Taxonomy records are versioned inputs to all later stages. V4 keeps taxonomy acquisition separate from cohort selection: the same normalized records can support human, livestock, crop, or mixed runs.

This notebook checks the pinned NCBI host taxonomy, inspects the converted pathogen taxonomy, reports group-level counts, and checks alias/variant references.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, log_step
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('01_taxonomy_hosts', RUN_ID)
print('Project root:', ROOT, '| log:', LOG_PATH)

## Step 1 · Inspect the pinned host taxonomy release

In [ ]:
import json
from IPython.display import display
manifest_path = ROOT / 'data/source_taxonomy/ncbi/source_manifest.json'
manifest = json.loads(manifest_path.read_text())
print('Manifest:', manifest_path)
display({k: manifest.get(k) for k in ('release', 'archive', 'sha256', 'retrieved_at') if k in manifest})
LOG.info('Loaded host taxonomy manifest: %s', manifest_path)

## Step 2 · Load the normalized V4 taxonomy tables

In [ ]:
import pandas as pd
STAGING_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1].parent
TAXONOMY_ROOT = STAGING_ROOT / 'taxonomy'
hosts = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
pathogens = pd.read_parquet(TAXONOMY_ROOT / 'pathogens.parquet')
aliases = pd.read_parquet(TAXONOMY_ROOT / 'pathogen_aliases.parquet')
variants = pd.read_parquet(TAXONOMY_ROOT / 'pathogen_variants.parquet')
print('Staging release:', STAGING_ROOT)
print('Loaded hosts:', len(hosts), '| pathogens:', len(pathogens), '| aliases:', len(aliases), '| variants:', len(variants))
LOG.info('Loaded taxonomy tables hosts=%s pathogens=%s aliases=%s variants=%s', len(hosts), len(pathogens), len(aliases), len(variants))

## Step 3 · Summarize pathogen groups and taxonomy sources

In [ ]:
display(pathogens.groupby(['pathogen_group', 'taxonomy_source'], dropna=False).size().rename('records').to_frame())
display(hosts[['host_id', 'accepted_name', 'host_group', 'taxonomy_release']].sort_values(['host_group', 'accepted_name']))
LOG.info('Pathogen group totals: %s', pathogens.groupby('pathogen_group').size().to_dict())

## Step 4 · Check normalized references

In [ ]:
pathogen_ids = set(pathogens.pathogen_id.astype(str))
linked_aliases = aliases.loc[aliases.pathogen_id.notna()] if len(aliases) else aliases
unknown_aliases = int((~linked_aliases.pathogen_id.astype(str).isin(pathogen_ids)).sum()) if len(linked_aliases) else 0
unresolved_aliases = int(aliases.pathogen_id.isna().sum()) if len(aliases) else 0
unknown_variants = int((~variants.pathogen_id.astype(str).isin(pathogen_ids)).sum()) if len(variants) else 0
TAXONOMY_CHECKS = {'unique_pathogen_ids': pathogens.pathogen_id.is_unique,
                   'unique_host_ids': hosts.host_id.is_unique,
                   'aliases_without_pathogen': unknown_aliases,
                   'unresolved_aliases_accounted': unresolved_aliases,
                   'variants_without_pathogen': unknown_variants}
display(TAXONOMY_CHECKS)
if unknown_aliases or unknown_variants or not pathogens.pathogen_id.is_unique or not hosts.host_id.is_unique:
    raise ValueError('Taxonomy reference checks failed; inspect the normalized tables before acquisition')
LOG.info('Taxonomy checks passed; unresolved aliases retained as unmapped rows: %d', unresolved_aliases)
print('Unresolved alias rows are retained with explicit missing pathogen IDs:', unresolved_aliases)

## Step 5 · Host identifiers available to run configuration

In [ ]:
HOST_LOOKUP = hosts[['host_id', 'accepted_name', 'host_group', 'taxon_id']].sort_values('host_id')
display(HOST_LOOKUP)
print('Use host_id values in run configuration; names and aliases are search terms.')
LOG.info('Host selectors available: %s', HOST_LOOKUP.host_id.tolist())